In [1]:
!pip install bertopic pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.6/112.6 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.0 requires scikit-learn<1.6.0,>=1.0.0, but you have scikit-learn 1.7.2 which is incompatible.


In [2]:
# !pip install keybert -q


In [3]:
# Data manipulation
import pandas as pd

# Topic modeling
from bertopic import BERTopic

# Visualization (optional, for later use)
import matplotlib.pyplot as plt
import plotly.express as px

# Text preprocessing (optional, for later use)
import re
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')  # Download stopwords if needed

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
from pathlib import Path

# Portable repository paths; works when notebook starts in root or notebooks/.
REPO_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_PATH = REPO_ROOT / "data" / "NEW DATASET 29 Studies.csv"
OUTPUT_DIR = REPO_ROOT / "results" / "run_04_umap_neighbors10"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

import pandas as pd

# Load the CSV (your encoding logic is fine)
try:
    df = pd.read_csv(str(DATA_PATH), encoding="utf-8-sig")
except UnicodeDecodeError:
    try:
        df = pd.read_csv(str(DATA_PATH), encoding="latin1")
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(str(DATA_PATH), encoding="ISO-8859-1")
        except UnicodeDecodeError:
            df = pd.read_csv(str(DATA_PATH), encoding="windows-1252")

# Clean column names (removes BOM/odd chars + trims spaces)
df.columns = df.columns.str.replace("\ufeff", "", regex=False).str.replace("ï»¿", "", regex=False).str.strip()

print("Columns in the dataset:", df.columns)

# Make sure columns exist (prevents silent key errors later)
required = ["Dimensions"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns in CSV: {missing}")

# Build docs using the single column
docs = (
    df["Dimensions"].fillna("").astype(str).str.strip()
).tolist()

print(f"Number of documents: {len(docs)}")
print("Sample documents:")
print(docs[:23])

Columns in the dataset: Index(['ID', 'Article title', 'Dimensions'], dtype='object')
Number of documents: 262
Sample documents:
['SOCIO-CULTURAL', 'TEACHING \x96 LEARNING', 'ACADEMIC MANAGEMENT', 'ADMINISTRATIVE MANAGEMENT', 'RESEARCH AND INNOVATION', 'DIGITAL GOVERNMENT', 'INSTITUTIONAL IMAGE AND DIGITAL MARKETING', 'UNIVERSITY EXTENSION', 'Sustainable Digital Technology Infrastructure', 'Sustainable Digital Operations and Processes', 'Sustainable Digital Strategy and Vision', 'Sustainable Innovation', 'Sustainable Digital Skills', 'Sustainable Digital Culture', 'Sustainable Digital Organization', 'Sustainable Governance and Management', 'Sustainable Digital Compliance and Security', 'Learning and Teaching Processes', 'Enabling Processes', 'Research Processes', 'Planning and Governance Processes', 'Strategy and culture', 'Financial resources']


In [5]:
import numpy as np
import pandas as pd
import spacy
import torch
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sentence_transformers import SentenceTransformer

# ------------------------------------------
# 0) Preconditions - Data Prep
# ------------------------------------------
df = df.copy()
df.columns = df.columns.astype(str).str.strip()

col_map = {c.lower(): c for c in df.columns}
if "dimensions" not in col_map:
    raise KeyError(f"'Dimensions' column not found. Available columns: {df.columns.tolist()}")

Dimensions_col = col_map["dimensions"]
Dimensions = df[Dimensions_col].fillna("").astype(str).str.strip()

docs = []
doc_idx = []
for i, code in enumerate(Dimensions):
    text = code.strip()
    if text:
        docs.append(text)
        doc_idx.append(i)

print(f"Using column: {Dimensions_col}")
print(f"Raw documents: {len(docs)}")

# ------------------------------------------
# 1) Preprocessing
# ------------------------------------------
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

def preprocess(texts, indices, batch_size=128):
    cleaned = []
    kept_idx = []
    for doc, idx in zip(nlp.pipe(texts, batch_size=batch_size), indices):
        toks = [t.lemma_.lower() for t in doc if not t.is_punct and not t.is_space]
        s = " ".join(toks).strip()
        if s:
            cleaned.append(s)
            kept_idx.append(idx)
    return cleaned, kept_idx

docs_cleaned, kept_idx = preprocess(docs, doc_idx)
print(f"Cleaned documents: {len(docs_cleaned)}")

if len(docs_cleaned) < 50:
    raise ValueError("Too few documents for modeling.")

# ------------------------------------------
# 2) Embeddings
# ------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

embeddings = embedding_model.encode(
    docs_cleaned,
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True
)

# ------------------------------------------
# 3) Vectorizer
# ------------------------------------------
custom_stop = list(set(ENGLISH_STOP_WORDS) - {"no", "not", "never"})

vectorizer_model = CountVectorizer(
    stop_words=custom_stop,
    ngram_range=(1, 2),
    min_df=1,
    max_df=1.0
)

# ------------------------------------------
# 4) Dimensionality Reduction
#    ⬆ Higher n_neighbors = more global structure = fewer broad topics
# ------------------------------------------
umap_model = UMAP(
    n_neighbors=10,       # was 4 → increase for fewer, broader topics
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# ------------------------------------------
# 5) Clustering
#    ⬆ Higher min_cluster_size = fewer but stronger clusters
# ------------------------------------------
hdbscan_model = HDBSCAN(
    min_cluster_size=10,  # was 4 → increase to merge small topics
    min_samples=3,        # was 1 → increase for more conservative clustering
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

# ------------------------------------------
# 6) BERTopic
#    nr_topics: auto-merges similar topics to reach your target count
#    ✅ Set your desired number of topics here (e.g. 8, 10, 12)
# ------------------------------------------
TARGET_TOPICS = 10  # 🔧 adjust this to your preferred number

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    nr_topics=TARGET_TOPICS,  # auto-merges down to this count after clustering
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs_cleaned, embeddings)

topic_info = topic_model.get_topic_info()
print("\nTopic Info:")
print(topic_info)

topics = np.array(topics)
outliers = (topics == -1)
outlier_rate = float(np.sum(outliers) / len(topics))
num_outliers = int(np.sum(outliers))
print(f"Outlier rate: {outlier_rate:.1%} ({num_outliers}/{len(topics)} docs)")

valid_topics = topics[topics >= 0]
num_topics = len(np.unique(valid_topics)) if len(valid_topics) > 0 else 0

# ------------------------------------------
# 7) Optional: further reduce AFTER fitting if still too many
#    Uncomment and set your desired count if needed
# ------------------------------------------
# if num_topics > TARGET_TOPICS:
#     topics, probs = topic_model.reduce_topics(docs_cleaned, nr_topics=TARGET_TOPICS)
#     topic_info = topic_model.get_topic_info()
#     print("\nReduced Topic Info:")
#     print(topic_info)

# ------------------------------------------
# 8) Map back to df
# ------------------------------------------
df["Topic"] = np.nan
df.loc[kept_idx, "Topic"] = topics

print(f"\n✅ SUCCESS! {num_topics} topics created")
print("df['Topic'] updated successfully!")
print(f"Topic distribution:\n{pd.Series(topics).value_counts().sort_index()}")

Using column: Dimensions
Raw documents: 188
Cleaned documents: 188


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

2026-11-06 11:22:10,983 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-06 11:22:21,013 - BERTopic - Dimensionality - Completed ✓
2026-11-06 11:22:21,015 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-06 11:22:21,031 - BERTopic - Cluster - Completed ✓
2026-11-06 11:22:21,032 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-11-06 11:22:21,094 - BERTopic - Representation - Completed ✓
2026-11-06 11:22:21,095 - BERTopic - Topic reduction - Reducing number of topics
2026-11-06 11:22:21,095 - BERTopic - Topic reduction - Number of topics (10) is equal or higher than the clustered topics(10).
2026-11-06 11:22:21,096 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-06 11:22:21,179 - BERTopic - Representation - Completed ✓



Topic Info:
   Topic  Count                                               Name  \
0     -1      8  -1_service_service society_society_transfer se...   
1      0     11        0_quality_assurance_datum_quality assurance   
2      1     24        1_teaching_learning_learning teaching_learn   
3      2     21  2_governance_planning_leadership_planning mana...   
4      3     23  3_infrastructure_technological_ict_technologic...   
5      4     12     4_digital_competence_digital teaching_literacy   
6      5     16  5_process_management_administrative_process ma...   
7      6     26  6_research_scientific research_scientific_know...   
8      7     32  7_digital_sustainable_sustainable digital_infr...   
9      8     15      8_culture_ict culture_culture digital_digital   

                                      Representation  \
0  [service, service society, society, transfer s...   
1  [quality, assurance, datum, quality assurance,...   
2  [teaching, learning, learning teaching, learn

In [6]:
# Visualize top words for topics in a bar chart
topic_model.visualize_barchart(width=290, height=320, top_n_topics=42, n_words=100)


In [7]:
# Heatmap without outlier topic (-1)
topic_ids = [t for t in topic_model.get_topic_info()["Topic"].tolist() if t != -1]
fig = topic_model.visualize_heatmap(topics=topic_ids)
fig.show()

In [8]:
# Get all topic info
topic_info = topic_model.get_topic_info()

# Print TF-IDF scores for each topic
for topic_id in topic_info["Topic"]:
    if topic_id == -1:
        continue  # skip outlier topic

    print(f"\n🟩 Topic {topic_id} (Count: {topic_info[topic_info['Topic'] == topic_id]['Count'].values[0]})")
    print("Top Words with TF-IDF Scores:")
    
    for word, score in topic_model.get_topic(topic_id):
        print(f"  {word:<20} {score:.4f}")



🟩 Topic 0 (Count: 11)
Top Words with TF-IDF Scores:
  quality              0.4233
  assurance            0.3748
  datum                0.3748
  quality assurance    0.3748
  assurance quality    0.1952
  assurance datum      0.1952
  datum quality        0.1952
  assurance persistent 0.1151
  datum literacy       0.1151
  datum datum          0.1151

🟩 Topic 1 (Count: 24)
Top Words with TF-IDF Scores:
  teaching             0.2232
  learning             0.2069
  learning teaching    0.1675
  learn                0.1436
  teaching learning    0.1262
  teaching learn       0.1073
  process learning     0.0939
  student              0.0939
  learner              0.0698
  learning learning    0.0698

🟩 Topic 2 (Count: 21)
Top Words with TF-IDF Scores:
  governance           0.2118
  planning             0.2049
  leadership           0.1716
  planning management  0.1507
  leadership planning  0.1507
  strategy             0.1344
  management           0.1158
  planning governance  0.1055
 

In [9]:
import pandas as pd

# Build a table mapping each document/code to its assigned topic
doc_df = pd.DataFrame({
    "code_raw": docs,             # your original codes (recommended to inspect)
    "code_cleaned": docs_cleaned, # cleaned version used for modeling
    "topic": topics
})

# Example: show all codes assigned to Topic 0
topic_id = 0
topic_docs = doc_df[doc_df["topic"] == topic_id]["code_raw"].tolist()

print("Docs in topic", topic_id, "=", len(topic_docs))
print("Sample docs:")
print(topic_docs[:190])  # first 20

Docs in topic 0 = 11
Sample docs:
['Quality Assurance', 'quality assurance', 'Quality assurance', 'Persistent data', 'Data quality', 'Quality Assurance', 'Data', 'Quality assurance', 'Data management', 'Data literacy', 'Assessment']


In [10]:
import pandas as pd

# Build a table mapping each document/code to its assigned topic
doc_df = pd.DataFrame({
    "code_raw": docs,             # your original codes (recommended to inspect)
    "code_cleaned": docs_cleaned, # cleaned version used for modeling
    "topic": topics
})

# Show all codes for ALL topics (instead of just one topic_id)
for topic_id in sorted(doc_df["topic"].unique()):
    topic_docs = doc_df[doc_df["topic"] == topic_id]["code_raw"].tolist()
    print("\n" + "="*60)
    print("Docs in topic", topic_id, "=", len(topic_docs))
    print("Dimensions:")
    print(topic_docs)   # prints all codes in that topic


Docs in topic -1 = 8
Dimensions:
['Technology Transfer and Service to Society', 'service Society', 'Technology transfer and service to the society', 'Technology Transfer and Service to Society', 'Technology transfer and service to the society', 'Research, networksand alliances', 'Professional services', 'Professional Engagement']

Docs in topic 0 = 11
Dimensions:
['Quality Assurance', 'quality assurance', 'Quality assurance', 'Persistent data', 'Data quality', 'Quality Assurance', 'Data', 'Quality assurance', 'Data management', 'Data literacy', 'Assessment']

Docs in topic 1 = 24
Dimensions:
['TEACHING \x96 LEARNING', 'Learning and Teaching Processes', 'Focus on the student experience', 'Teaching-learning processes', 'Learning and Teaching', 'Learning and teaching', 'Curriculum and learning', 'Student services and other users', 'Teaching-Learning', 'Learning and Teaching', 'Focus on the students', 'Teaching and learning processes', 'Learning and Teaching Process', 'Learning and teachi

In [11]:
import pandas as pd
from IPython.display import display, HTML

# Build a table mapping each document/code to its assigned topic
doc_df = pd.DataFrame({
    "code_raw": docs,
    "code_cleaned": docs_cleaned,
    "topic": topics
})

# --- Create a "wide" table: one column per topic, rows are the codes in that topic ---
topic_lists = {
    t: doc_df.loc[doc_df["topic"] == t, "code_raw"].tolist()
    for t in sorted(doc_df["topic"].unique())
}
max_len = max(len(v) for v in topic_lists.values())

topic_table = pd.DataFrame({
    f"T{t} (n={len(v)})": v + [""] * (max_len - len(v))
    for t, v in topic_lists.items()
})

# ---- Smaller, screenshot-friendly table styling (no truncation of rows) ----
styled = (topic_table.style
          .set_table_styles([
              {"selector": "table", "props": [("font-size", "9px"), ("border-collapse", "collapse")]},
              {"selector": "th",    "props": [("font-size", "9px"), ("padding", "2px 4px"), ("text-align", "left")]},
              {"selector": "td",    "props": [("font-size", "9px"), ("padding", "2px 4px"), ("vertical-align", "top")]}
          ])
          .set_properties(**{"white-space": "normal"})
         )

display(styled)

,T-1 (n=8),T0 (n=11),T1 (n=24),T2 (n=21),T3 (n=23),T4 (n=12),T5 (n=16),T6 (n=26),T7 (n=32),T8 (n=15)
0,Technology Transfer and Service to Society,Quality Assurance,TEACHING  LEARNING,Sustainable Governance and Management,SOCIO-CULTURAL,digital teaching and learning,ACADEMIC MANAGEMENT,RESEARCH AND INNOVATION,DIGITAL GOVERNMENT,INSTITUTIONAL IMAGE AND DIGITAL MARKETING
1,service Society,quality assurance,Learning and Teaching Processes,Planning and Governance Processes,Technological basis,Online forms,ADMINISTRATIVE MANAGEMENT,Research Processes,Sustainable Digital Technology Infrastructure,ICT Culture
2,Technology transfer and service to the society,Quality assurance,Focus on the student experience,Strategy and culture,technical and technological,Digital Education,UNIVERSITY EXTENSION,Competencies and abilities,Sustainable Digital Operations and Processes,technology culture
3,Technology Transfer and Service to Society,Persistent data,Teaching-learning processes,IT governance,cognitive,Wide use of innovative digital technologies in Teaching and Learning,Enabling Processes,Scientific Research Work,Sustainable Digital Strategy and Vision,ICT culture
4,Technology transfer and service to the society,Data quality,Learning and Teaching,Leadership & Culture,social,Rich Digital Content and presence of digital skills and competences in curricula,Economic-administrative processes,scientific research,Sustainable Innovation,No media discontinuity
5,"Research, networksand alliances",Quality Assurance,Learning and teaching,"Leadership, Planning, and Management",psychological,Provision of digital Training and Support to stakeholders,Internal process management,Scientific-research work,Sustainable Digital Skills,Institutional Image-Mkt
6,Professional services,Data,Curriculum and learning,"leadership, planning and management",spiritual,Digital proficiency,Administration and management,Research and innovation,Sustainable Digital Culture,Digital Culture
7,Professional Engagement,Quality assurance,Student services and other users,"Leadership, planning, and management",Strengthened Logistics,Digital learning,Parallel processing,Paper,Sustainable Digital Organization,ICT culture
8,,Data management,Teaching-Learning,"Leadership, Planning, and Management",ICT Resources and Infrastructure,Digital teaching,Academic Management,Help,Sustainable Digital Compliance and Security,Digital Culture
9,,Data literacy,Learning and Teaching,Strategy and culture,technology and infrastructure,Information literacy,Administrative Management,Exceptions,Financial resources,Digital Research and Communities


In [12]:
import pandas as pd
from IPython.display import display, HTML

# Build a table mapping each document/code to its assigned topic
doc_df = pd.DataFrame({
    "code_raw": docs,
    "code_cleaned": docs_cleaned,
    "topic": topics
})

# --- Create a "wide" table: one column per topic, rows are the codes in that topic ---
topic_lists = {
    t: doc_df.loc[doc_df["topic"] == t, "code_raw"].tolist()
    for t in sorted(doc_df["topic"].unique())
}
max_len = max(len(v) for v in topic_lists.values())

topic_table = pd.DataFrame({
    f"T{t} (n={len(v)})": v + [""] * (max_len - len(v))
    for t, v in topic_lists.items()
})

# ---- Smaller, screenshot-friendly table styling (no truncation of rows) ----
styled = (topic_table.style
          .set_table_styles([
              {"selector": "table", "props": [("font-size", "9px"), ("border-collapse", "collapse")]},
              {"selector": "th",    "props": [("font-size", "9px"), ("padding", "2px 4px"), ("text-align", "left")]},
              {"selector": "td",    "props": [("font-size", "9px"), ("padding", "2px 4px"), ("vertical-align", "top")]}
          ])
          .set_properties(**{"white-space": "normal"})
         )

display(styled)

# -----------------------------
# SAVE OUTPUT TO FILES
# -----------------------------

# 1) Save the wide table as CSV (best for NVivo/Excel)
topic_table.to_csv(OUTPUT_DIR / "topic_table_wide.csv", index=False)

# 2) Save the styled table as HTML (keeps the formatting)
styled.to_html(OUTPUT_DIR / "topic_table_wide_styled.html")

print("Saved files:")
print("- topic_table_wide.csv")
print("- topic_table_wide_styled.html")

,T-1 (n=8),T0 (n=11),T1 (n=24),T2 (n=21),T3 (n=23),T4 (n=12),T5 (n=16),T6 (n=26),T7 (n=32),T8 (n=15)
0,Technology Transfer and Service to Society,Quality Assurance,TEACHING  LEARNING,Sustainable Governance and Management,SOCIO-CULTURAL,digital teaching and learning,ACADEMIC MANAGEMENT,RESEARCH AND INNOVATION,DIGITAL GOVERNMENT,INSTITUTIONAL IMAGE AND DIGITAL MARKETING
1,service Society,quality assurance,Learning and Teaching Processes,Planning and Governance Processes,Technological basis,Online forms,ADMINISTRATIVE MANAGEMENT,Research Processes,Sustainable Digital Technology Infrastructure,ICT Culture
2,Technology transfer and service to the society,Quality assurance,Focus on the student experience,Strategy and culture,technical and technological,Digital Education,UNIVERSITY EXTENSION,Competencies and abilities,Sustainable Digital Operations and Processes,technology culture
3,Technology Transfer and Service to Society,Persistent data,Teaching-learning processes,IT governance,cognitive,Wide use of innovative digital technologies in Teaching and Learning,Enabling Processes,Scientific Research Work,Sustainable Digital Strategy and Vision,ICT culture
4,Technology transfer and service to the society,Data quality,Learning and Teaching,Leadership & Culture,social,Rich Digital Content and presence of digital skills and competences in curricula,Economic-administrative processes,scientific research,Sustainable Innovation,No media discontinuity
5,"Research, networksand alliances",Quality Assurance,Learning and teaching,"Leadership, Planning, and Management",psychological,Provision of digital Training and Support to stakeholders,Internal process management,Scientific-research work,Sustainable Digital Skills,Institutional Image-Mkt
6,Professional services,Data,Curriculum and learning,"leadership, planning and management",spiritual,Digital proficiency,Administration and management,Research and innovation,Sustainable Digital Culture,Digital Culture
7,Professional Engagement,Quality assurance,Student services and other users,"Leadership, planning, and management",Strengthened Logistics,Digital learning,Parallel processing,Paper,Sustainable Digital Organization,ICT culture
8,,Data management,Teaching-Learning,"Leadership, Planning, and Management",ICT Resources and Infrastructure,Digital teaching,Academic Management,Help,Sustainable Digital Compliance and Security,Digital Culture
9,,Data literacy,Learning and Teaching,Strategy and culture,technology and infrastructure,Information literacy,Administrative Management,Exceptions,Financial resources,Digital Research and Communities


Saved files:
- topic_table_wide.csv
- topic_table_wide_styled.html


In [13]:
# Visualize Documents (like the BERTopic docs example)
# Assumes you already have: topic_model, docs_cleaned, embeddings

from umap import UMAP

# Option 1 (recommended): pre-reduce embeddings (faster + stable for screenshots)
reduced_embeddings = UMAP(
    n_neighbors=10,
    n_components=2,
    min_dist=0.0,
    metric="cosine",
    random_state=42
).fit_transform(embeddings)

fig = topic_model.visualize_documents(
    docs_cleaned,
    reduced_embeddings=reduced_embeddings
)
fig.show()

In [14]:
# Visualize Documents (make dots bigger) — works even if BERTopic has no marker_size arg
from umap import UMAP

reduced_embeddings = UMAP(
    n_neighbors=10,
    n_components=2,
    min_dist=0.0,
    metric="cosine",
    random_state=42
).fit_transform(embeddings)

fig = topic_model.visualize_documents(
    docs_cleaned,
    reduced_embeddings=reduced_embeddings
)

# ✅ Make the points bigger (try 10, 12, 15, 20)
fig.update_traces(marker=dict(size=14))

fig.show()

In [15]:
import numpy as np
print("Outliers:", sum(np.array(topics) == -1))


Outliers: 8


In [16]:
# Make sure you have these variables already:
# - docs_cleaned: your list of preprocessed documents
# - topics: list of topic assignments (includes -1 for outliers)
# - topic_model: your trained BERTopic model

from collections import defaultdict

# Step 1: Group documents by their assigned topic
clustered_docs = defaultdict(list)
for doc, topic in zip(docs_cleaned, topics):
    clustered_docs[topic].append(doc)

# Step 2: Print documents and their words by cluster
for topic_id in sorted(clustered_docs.keys()):
    if topic_id == -1:
        print("\n🟥 Outlier Documents (Topic -1):")
    else:
        print(f"\n🟩 Topic {topic_id} — Top Words:")
        top_words = topic_model.get_topic(topic_id)
        for word, score in top_words:
            print(f"  {word:<20} {score:.4f}")
    
    print("📄 Documents:")
    for doc in clustered_docs[topic_id]:
        print(f"  - {doc}")



🟥 Outlier Documents (Topic -1):
📄 Documents:
  - technology transfer and service to society
  - service society
  - technology transfer and service to the society
  - technology transfer and service to society
  - technology transfer and service to the society
  - research networksand alliance
  - professional service
  - professional engagement

🟩 Topic 0 — Top Words:
  quality              0.4233
  assurance            0.3748
  datum                0.3748
  quality assurance    0.3748
  assurance quality    0.1952
  assurance datum      0.1952
  datum quality        0.1952
  assurance persistent 0.1151
  datum literacy       0.1151
  datum datum          0.1151
📄 Documents:
  - quality assurance
  - quality assurance
  - quality assurance
  - persistent datum
  - datum quality
  - quality assurance
  - datum
  - quality assurance
  - datum management
  - datum literacy
  - assessment

🟩 Topic 1 — Top Words:
  teaching             0.2232
  learning             0.2069
  learning teach

In [17]:
import numpy as np
import pandas as pd

topics_arr = np.asarray(topics).astype(int)

# ✅ HDBSCAN already provides soft membership probabilities
# topic_model.hdbscan_model.probabilities_ shape = (n_docs,)
# This is the probability of each doc belonging to its assigned cluster
hdbscan_probs = topic_model.hdbscan_model.probabilities_

# Build summary table
conf_df = pd.DataFrame({
    "topic": topics_arr,
    "prob": hdbscan_probs
})

# Optional: exclude outliers (topic == -1)
conf_df = conf_df[conf_df["topic"] != -1]

topic_conf = (
    conf_df.groupby("topic")["prob"]
    .agg(
        n="size",
        mean="mean",
        median="median",
        min="min",
        max="max"
    )
    .reset_index()
)

topic_conf["pct"] = (topic_conf["n"] / len(conf_df) * 100).round(2)
topic_conf = topic_conf.sort_values("mean", ascending=False).reset_index(drop=True)
print(topic_conf)

   topic   n      mean    median       min  max    pct
0      0  11  1.000000  1.000000  1.000000  1.0   6.11
1      5  16  0.983368  1.000000  0.733882  1.0   8.89
2      8  15  0.976390  1.000000  0.925199  1.0   8.33
3      4  12  0.942813  1.000000  0.615368  1.0   6.67
4      1  24  0.936490  1.000000  0.480790  1.0  13.33
5      2  21  0.905708  1.000000  0.714915  1.0  11.67
6      7  32  0.891849  1.000000  0.657083  1.0  17.78
7      3  23  0.879722  1.000000  0.692623  1.0  12.78
8      6  26  0.850454  0.938947  0.591677  1.0  14.44


In [18]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import mannwhitneyu

rng = np.random.default_rng(42)

# sample pairs
idx = np.where(topics != -1)[0]
sample_size = min(5000, len(idx))  # adjust
pairs = rng.choice(idx, size=(sample_size, 2), replace=True)

def same_topic(i, j):
    return topics[i] == topics[j]

within = []
between = []
for i, j in pairs:
    sim = float(cosine_similarity(embeddings[i].reshape(1,-1),
                                  embeddings[j].reshape(1,-1))[0,0])
    (within if same_topic(i, j) else between).append(sim)

within = np.array(within)
between = np.array(between)

stat, p = mannwhitneyu(within, between, alternative="greater")
print("within mean:", within.mean(), "between mean:", between.mean())
print("Mann–Whitney U p-value:", p)

within mean: 0.6045954417851236 between mean: 0.250183442607522
Mann–Whitney U p-value: 5.356032194561107e-08


In [19]:

# Save topic assignments for stability comparison
assignments = pd.DataFrame({
    "row_index": kept_idx,
    "construct_label": docs_cleaned,
    "topic_id": topics
})

assignments.to_csv(
    OUTPUT_DIR / "topic_assignments_baseline.csv",
    index=False
)

topic_model.get_topic_info().to_csv(
    OUTPUT_DIR / "topic_summary_baseline.csv",
    index=False
)

print("Baseline assignments and topic summary saved.")


Baseline assignments and topic summary saved.
